In [5]:
# Cell 1: Setup & Imports
import asyncio
import asyncpg
import os
from dotenv import load_dotenv
import pandas as pd
import json


In [6]:
load_dotenv()

True

In [8]:

load_dotenv()

# Get database connection string
DATABASE_URL = os.getenv("DATABASE_URL")

# Cell 2: Function to fetch farmer data by farmer_id
async def get_farmer_details(farmer_id: str):
    """
    Fetch farmer details from farmers table using farmer_id
    Returns: farmer profile with all related farm and prediction data
    """
    try:
        conn = await asyncpg.connect(DATABASE_URL)
        
        # Fetch farmer profile
        farmer = await conn.fetchrow(
            "SELECT id, name, phone, state_name, dist_name, created_at FROM farmers WHERE id = $1",
            farmer_id
        )
        
        if not farmer:
            print(f"❌ Farmer with ID '{farmer_id}' not found")
            await conn.close()
            return None
        
        # Fetch farm fields
        fields = await conn.fetch(
            """SELECT id, field_name, city_name, state_name, area_hectares, 
                      center_lat, center_lon, polygon_id, created_at
               FROM farm_fields WHERE farmer_id = $1 ORDER BY created_at DESC""",
            farmer_id
        )
        
        # Fetch latest prediction for each field
        predictions = []
        for field in fields:
            pred = await conn.fetchrow(
                """SELECT crop_type, npk_input, irrigation_ratio, ndvi_value, 
                          final_health_score, predicted_yield, risk_level, year
                   FROM field_predictions WHERE field_id = $1 
                   ORDER BY year DESC, calculated_at DESC LIMIT 1""",
                str(field['id'])
            )
            predictions.append(pred)
        
        await conn.close()
        
        return {
            "farmer": dict(farmer),
            "fields": [dict(f) for f in fields],
            "predictions": [dict(p) if p else None for p in predictions]
        }
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return None


result = await get_farmer_details(farmer_id)

if result:
    print("✅ FARMER PROFILE")
    print("=" * 60)
    df_farmer = pd.DataFrame([result['farmer']])
    print(df_farmer.to_string())
    
    print("\n✅ FARM FIELDS")
    print("=" * 60)
    if result['fields']:
        df_fields = pd.DataFrame(result['fields'])
        print(df_fields.to_string())
    else:
        print("No farm fields registered")
    
    print("\n✅ LATEST PREDICTIONS")
    print("=" * 60)
    for i, pred in enumerate(result['predictions']):
        if pred:
            print(f"\nField {i+1} Prediction:")
            for key, value in pred.items():
                print(f"  {key}: {value}")
        else:
            print(f"Field {i+1}: No predictions yet")

✅ FARMER PROFILE
                                     id         name           phone     state_name dist_name                 created_at
0  c59f6f44-1a98-4eaa-8cf0-3581316a32bb  Arjun Yadav  +91-9876543100  Uttar Pradesh      agra 2026-04-23 10:11:24.169394

✅ FARM FIELDS
                                     id           field_name city_name     state_name  area_hectares  center_lat  center_lon                polygon_id                 created_at
0  b354f706-c77f-4cc6-bfc6-27773b1e5626  Yamuna Khadar Field      Agra  Uttar Pradesh            2.5     27.1807     78.0204  69e9f050d02279628247ba9d 2026-04-23 10:11:29.326465

✅ LATEST PREDICTIONS

Field 1 Prediction:
  crop_type: WHEAT.YIELD.Kg.per.ha.
  npk_input: 120.0
  irrigation_ratio: 0.85
  ndvi_value: 0.21113329071512726
  final_health_score: 65.83
  predicted_yield: 59873.14171519782
  risk_level: LOW
  year: 2025


In [7]:
# 🔍 ERROR DEBUGGING CELL
# Run this to see the exact error
# Cell 3: Test with farmer_id
farmer_id = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb"  # Replace with actual farmer_id

async def debug_get_farmer_details(farmer_id: str):
    """
    Debug version with detailed error messages
    """
    print(f"🔍 Debugging farmer_id: {farmer_id}")
    print(f"🔍 DATABASE_URL exists: {DATABASE_URL is not None}")
    
    try:
        print("📡 Attempting database connection...")
        conn = await asyncpg.connect(DATABASE_URL)
        print("✅ Database connection successful!")
        
        print(f"🔍 Querying farmer with ID: {farmer_id}")
        farmer = await conn.fetchrow(
            "SELECT id, name, phone, state_name, dist_name, created_at FROM farmers WHERE id = $1",
            farmer_id
        )
        
        if not farmer:
            print(f"❌ ERROR: Farmer with ID '{farmer_id}' NOT FOUND in database")
            await conn.close()
            return None
        
        print(f"✅ Found farmer: {farmer['name']}")
        
        await conn.close()
        return farmer
        
    except ValueError as e:
        print(f"❌ ERROR - UUID PARSING ERROR: {e}")
        print(f"   Invalid farmer_id format. Make sure it's a valid UUID.")
        return None
    except asyncpg.exceptions.PostgresError as e:
        print(f"❌ ERROR - DATABASE ERROR: {e}")
        print(f"   Error Type: {type(e).__name__}")
        return None
    except Exception as e:
        print(f"❌ ERROR - UNEXPECTED ERROR: {e}")
        print(f"   Error Type: {type(e).__name__}")
        import traceback
        traceback.print_exc()
        return None

# Test it
debug_result = await debug_get_farmer_details(farmer_id)
print(f"\n📊 Debug Result: {debug_result}")


🔍 Debugging farmer_id: c59f6f44-1a98-4eaa-8cf0-3581316a32bb


NameError: name 'DATABASE_URL' is not defined

In [4]:
# 📋 CHECK ALL FARMERS IN DATABASE
async def list_all_farmers():
    """
    Shows all farmers in the database
    """
    try:
        conn = await asyncpg.connect(DATABASE_URL)
        farmers = await conn.fetch(
            "SELECT id, name, phone, state_name, dist_name FROM farmers LIMIT 20"
        )
        await conn.close()
        
        if not farmers:
            print("❌ No farmers found in database. Please register a farmer first.")
            return None
        
        print(f"✅ Found {len(farmers)} farmer(s):\n")
        df = pd.DataFrame([dict(f) for f in farmers])
        print(df.to_string())
        print("\n📌 Copy a farmer ID from above and use it in farmer_id variable")
        
        return farmers
        
    except Exception as e:
        print(f"❌ ERROR: {e}")
        return None

all_farmers = await list_all_farmers()


❌ ERROR: name 'asyncpg' is not defined


### WE will get the final_things as Different_Mandi+State+City

In [8]:
# 🌾 GET MANDI BY FARMER_ID
import json

# Load mandi master data
MANDI_JSON_PATH = "D:\CA_content\Python\KissanSathi\krishisarthi-api\mandi_master.json"

try:
    with open(MANDI_JSON_PATH, 'r') as f:
        mandi_data = json.load(f)
    print("✅ Mandi master data loaded successfully")
except Exception as e:
    print(f"❌ Error loading mandi data: {e}")
    mandi_data = None


async def get_farmer_with_mandi(farmer_id: str):
    """
    Fetch farmer details and get corresponding mandi from state & city
    """
    try:
        conn = await asyncpg.connect(DATABASE_URL)
        
        # Get farmer's state and city
        farmer = await conn.fetchrow(
            """SELECT id, name, phone, state_name, dist_name 
               FROM farmers WHERE id = $1""",
            farmer_id
        )
        
        if not farmer:
            print(f"❌ Farmer with ID '{farmer_id}' not found")
            await conn.close()
            return None
        
        await conn.close()
        
        # Get mandi for this state and city
        state = farmer['state_name']
        city = farmer['dist_name']
        
        mandi_list = []
        if mandi_data:
            # Case-insensitive lookup for state
            target_state_data = None
            for s_key in mandi_data:
                if s_key.lower() == state.lower():
                    target_state_data = mandi_data[s_key]
                    break
            
            if target_state_data:
                # Case-insensitive lookup for city
                for c_key in target_state_data:
                    if c_key.lower() == city.lower():
                        mandi_list = target_state_data[c_key]
                        break
        
        return {
            "farmer_id": str(farmer['id']),
            "farmer_name": farmer['name'],
            "phone": farmer['phone'],
            "state": state,
            "city": city,
            "mandis": mandi_list
        }
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
        return None


# 🔍 TEST WITH FARMER_ID
print("\n" + "="*70)
print("🌾 FETCHING FARMER DETAILS WITH MANDI")
print("="*70)

# Use an existing farmer_id from your database
test_farmer_id = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb"  # Replace with actual ID

result = await get_farmer_with_mandi(test_farmer_id)

if result:
    print(f"\n✅ FARMER PROFILE")
    print("-" * 70)
    print(f"  Farmer ID    : {result['farmer_id']}")
    print(f"  Name         : {result['farmer_name']}")
    print(f"  Phone        : {result['phone']}")
    print(f"  State        : {result['state']}")
    print(f"  City/District: {result['city']}")
    
    print(f"\n🏪 MANDIS IN {result['city']}, {result['state']}")
    print("-" * 70)
    if result['mandis']:
        for i, mandi in enumerate(result['mandis'], 1):
            print(f"  {i}. {mandi}")
    else:
        print(f"  ⚠️  No mandis found for {result['city']}")
        # Show available cities in this state
        if result['state'] in mandi_data:
            print(f"\n📍 Available cities in {result['state']}:")
            for available_city in list(mandi_data[result['state']].keys())[:10]:
                print(f"     - {available_city}")
else:
    print("❌ Could not fetch data")

<>:5: SyntaxWarning: invalid escape sequence '\C'
C:\Users\asus\AppData\Local\Temp\ipykernel_9412\1294526867.py:5: SyntaxWarning: invalid escape sequence '\C'
  MANDI_JSON_PATH = "D:\CA_content\Python\KissanSathi\krishisarthi-api\mandi_master.json"


✅ Mandi master data loaded successfully

🌾 FETCHING FARMER DETAILS WITH MANDI

✅ FARMER PROFILE
----------------------------------------------------------------------
  Farmer ID    : c59f6f44-1a98-4eaa-8cf0-3581316a32bb
  Name         : Arjun Yadav
  Phone        : +91-9876543100
  State        : Uttar Pradesh
  City/District: agra

🏪 MANDIS IN agra, Uttar Pradesh
----------------------------------------------------------------------
  ⚠️  No mandis found for agra

📍 Available cities in Uttar Pradesh:
     - Auraiya
     - Chitrakut
     - Balrampur
     - Jalaun (Orai)
     - Sitapur
     - Aligarh
     - Ayodhya
     - Azamgarh
     - Badaun
     - Bahraich


In [9]:
async def get_farmer_with_mandi(farmer_id: str):
    """
    Fetch farmer details and get corresponding mandi from state & city (Case-Insensitive)
    """
    try:
        conn = await asyncpg.connect(DATABASE_URL)
        
        # Get farmer's state and city
        farmer = await conn.fetchrow(
            """SELECT id, name, phone, state_name, dist_name 
               FROM farmers WHERE id = $1""",
            farmer_id
        )
        
        if not farmer:
            print(f"❌ Farmer with ID '{farmer_id}' not found")
            await conn.close()
            return None
        
        await conn.close()
        
        # Get mandi for this state and city
        state = farmer['state_name']
        city = farmer['dist_name']
        
        mandi_list = []
        if mandi_data:
            # Step 1: Case-insensitive lookup for State
            target_state_data = None
            for s_key in mandi_data:
                if s_key.lower() == state.lower():
                    target_state_data = mandi_data[s_key]
                    break
            
            if target_state_data:
                # Step 2: Case-insensitive lookup for City/District
                for c_key in target_state_data:
                    if c_key.lower() == city.lower():
                        mandi_list = target_state_data[c_key]
                        break
        
        return {
            "farmer_id": str(farmer['id']),
            "farmer_name": farmer['name'],
            "phone": farmer['phone'],
            "state": state,
            "city": city,
            "mandis": mandi_list
        }
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return None


In [10]:


# 🔍 TEST WITH FARMER_ID
print("\n" + "="*70)
print("🌾 FETCHING FARMER DETAILS WITH MANDI")
print("="*70)

# Use an existing farmer_id from your database
test_farmer_id = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb"  # Replace with actual ID

result = await get_farmer_with_mandi(test_farmer_id)

if result:
    print(f"\n✅ FARMER PROFILE")
    print("-" * 70)
    print(f"  Farmer ID    : {result['farmer_id']}")
    print(f"  Name         : {result['farmer_name']}")
    print(f"  Phone        : {result['phone']}")
    print(f"  State        : {result['state']}")
    print(f"  City/District: {result['city']}")
    
    print(f"\n🏪 MANDIS IN {result['city']}, {result['state']}")
    print("-" * 70)
    if result['mandis']:
        for i, mandi in enumerate(result['mandis'], 1):
            print(f"  {i}. {mandi}")
    else:
        print(f"  ⚠️  No mandis found for {result['city']}")
        # Show available cities in this state
        if result['state'] in mandi_data:
            print(f"\n📍 Available cities in {result['state']}:")
            for available_city in list(mandi_data[result['state']].keys())[:10]:
                print(f"     - {available_city}")
else:
    print("❌ Could not fetch data")


🌾 FETCHING FARMER DETAILS WITH MANDI

✅ FARMER PROFILE
----------------------------------------------------------------------
  Farmer ID    : c59f6f44-1a98-4eaa-8cf0-3581316a32bb
  Name         : Arjun Yadav
  Phone        : +91-9876543100
  State        : Uttar Pradesh
  City/District: agra

🏪 MANDIS IN agra, Uttar Pradesh
----------------------------------------------------------------------
  1. Achnera APMC
  2. Agra APMC
  3. Fatehabad APMC
  4. Fatehpur Sikri APMC
  5. Jagnair APMC
  6. Jarar APMC


In [12]:
import os
import json
import logging
import httpx
from langchain_core.tools import tool
from chatbot.db import get_db_connection

logger = logging.getLogger(__name__)

AGMARKNET_API_KEY = os.getenv("AGMARKNET_API_KEY", "")
AGMARKNET_URL     = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"

async def _get_farm_context(farmer_id: str) -> dict:
    """Helper to fetch farmer's location and crop if not provided."""
    async with get_db_connection() as conn:
        farm = await conn.fetchrow(
            "SELECT ff.city_name, ff.state_name, f.state_name AS farmer_state, f.dist_name AS farmer_dist "
            "FROM farm_fields ff JOIN farmers f ON f.id = ff.farmer_id "
            "WHERE ff.farmer_id = $1 ORDER BY ff.created_at DESC LIMIT 1",
            farmer_id,
        )
        season = await conn.fetchrow(
            "SELECT fp.crop_type FROM field_predictions fp "
            "JOIN farm_fields ff ON ff.id = fp.field_id "
            "WHERE ff.farmer_id = $1 ORDER BY fp.year DESC, fp.calculated_at DESC LIMIT 1",
            farmer_id,
        )

    state    = (farm["state_name"] if farm else None) or (farm["farmer_state"] if farm else None)
    district = (farm["farmer_dist"]  if farm else None) or (farm["city_name"]   if farm else None)
    crop     = season["crop_type"] if season else None

    # Clean crop name (e.g., 'WHEAT.YIELD.Kg.per.ha.' -> 'Wheat')
    clean_crop = (
        (crop or "")
        .replace(".Kg.per.ha.", "").replace("YIELD", "").replace(".", " ").strip().title()
    )

    return {
        "state": state or "Unknown", 
        "district": district or "Unknown", 
        "crop_display": clean_crop
    }

@tool
async def get_market_price(
    farmer_id: str,
    mandi_name: str = "",
    state: str = "",
    district: str = "",
    crop: str = "",
) -> dict:
    """
    Fetches current crop price (min, max, modal) from Agmarknet API for a specific mandi.

    If mandi_name is not provided, returns the list of available mandis in the farmer's
    district (or the explicit district provided) so the LLM can ask the farmer to choose one.

    For cross-district queries, pass explicit state and district values.
    If location is unknown and the farmer did not mention one, do NOT call this tool —
    ask the farmer for their district/state first.

    Args:
        farmer_id:  UUID of the farmer
        mandi_name: exact mandi name (optional — if empty, returns results for the district)
        state:      override state name (use when farmer asks about a different state)
        district:   override district name (use when farmer asks about a different district)
        crop:       override crop name for price query (use when farmer asks about a specific crop)
    """
    # 1. Resolve parameters from DB if they are missing
    farm_ctx = await _get_farm_context(farmer_id)
    
    eff_state    = state.strip()    or farm_ctx["state"]
    eff_district = district.strip() or farm_ctx["district"]
    eff_crop     = crop.strip()     or farm_ctx["crop_display"]

    if not AGMARKNET_API_KEY:
        raise Exception("Configuration Error: AGMARKNET_API_KEY is missing in environment variables.")

    # 2. Build API Parameters
    params = {
        "api-key": AGMARKNET_API_KEY,
        "format": "json",
        "offset": 0,
        "limit": 10,
        "filters[state.keyword]": eff_state,
        "filters[commodity]": eff_crop,
    }
    
    if mandi_name.strip():
        params["filters[market]"] = mandi_name.strip()
    elif eff_district and eff_district != "Unknown":
        params["filters[district]"] = eff_district

    # 3. Perform API Call
    try:
        async with httpx.AsyncClient() as client:
            response = await client.get(AGMARKNET_URL, params=params, timeout=15.0)
            
        if response.status_code != 200:
            raise Exception(f"API Call Failed: Received status {response.status_code} from Agmarknet.")
            
        data = response.json()
        
    except Exception as e:
        logger.error(f"Market Price API Error: {str(e)}")
        raise Exception(f"Failed to fetch market data: {str(e)}")

    # 4. Check for Records
    if not data.get("records") or len(data["records"]) == 0:
        raise Exception(f"No price data found for {eff_crop} in {mandi_name or eff_district}, {eff_state}.")

    # 5. Return Result
    # If a specific mandi was requested, return the first record. Otherwise, return the full result set.
    return data["records"][0] if mandi_name else data


ModuleNotFoundError: No module named 'chatbot'

In [11]:
_get_farm_context("c59f6f44-1a98-4eaa-8cf0-3581316a32bb")

NameError: name '_get_farm_context' is not defined

In [14]:
import os
import json
import httpx
import asyncpg
import asyncio
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")
MANDI_JSON_PATH = os.getenv("MANDI_JSON_PATH", "mandi_master.json")
AGMARKNET_API_KEY = os.getenv("AGMARKNET_API_KEY")
AGMARKNET_URL = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"

class MandiPriceService:
    def __init__(self):
        self.mandi_data = self._load_mandi_data()

    def _load_mandi_data(self):
        """Loads the mandi_master.json file."""
        if not os.path.exists(MANDI_JSON_PATH):
            print(f"⚠️ Warning: {MANDI_JSON_PATH} not found.")
            return {}
        with open(MANDI_JSON_PATH, "r", encoding="utf-8") as f:
            return json.load(f)

    async def get_farmer_context(self, farmer_id: str):
        """Step 1: Fetch farmer's state and district from the database."""
        try:
            conn = await asyncpg.connect(DATABASE_URL)
            row = await conn.fetchrow(
                "SELECT state_name, dist_name FROM farmers WHERE id = $1",
                farmer_id
            )
            await conn.close()
            
            if not row:
                raise Exception(f"Farmer with ID {farmer_id} not found.")
                
            return {
                "state": row["state_name"],
                "district": row["dist_name"]
            }
        except Exception as e:
            raise Exception(f"Database Error: {str(e)}")

    def get_available_mandis(self, state: str, district: str):
        """Step 2: Get list of mandis from JSON using case-insensitive matching."""
        if not self.mandi_data:
            return []

        # Case-insensitive lookup
        target_state_data = None
        for s_key in self.mandi_data:
            if s_key.lower() == state.lower():
                target_state_data = self.mandi_data[s_key]
                break
        
        if target_state_data:
            for d_key in target_state_data:
                if d_key.lower() == district.lower():
                    return target_state_data[d_key]
        
        return []

    async def fetch_live_prices(self, state: str, mandi_name: str, crop: str):
        """Step 3: Call Agmarknet API and return the latest price result."""
        if not AGMARKNET_API_KEY:
            raise Exception("AGMARKNET_API_KEY is not set.")

        params = {
            "api-key": AGMARKNET_API_KEY,
            "format": "json",
            "filters[state.keyword]": state,
            "filters[market]": mandi_name,
            "filters[commodity]": crop
        }

        async with httpx.AsyncClient() as client:
            response = await client.get(AGMARKNET_URL, params=params, timeout=15.0)
        
        if response.status_code != 200:
            raise Exception(f"API Error: Status {response.status_code}")

        data = response.json()
        if not data.get("records"):
            raise Exception(f"No price records found for {crop} at {mandi_name}.")

        return data["records"][0]
# --- EXAMPLE USAGE ---
async def main():
    service = MandiPriceService()
    f_id = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb" 
    
    try:
        # 1. Get Location
        ctx = await service.get_farmer_context(f_id)
        print(f"📍 Location: {ctx['district']}, {ctx['state']}")
        
        # 2. Get Mandis
        mandis = service.get_available_mandis(ctx['state'], ctx['district'])
        print(f"🏪 Available Mandis: {mandis}")
        
        # 3. Fetch Prices
        if mandis:
            selected_mandi = mandis[0]
            price_data = await service.fetch_live_prices(ctx['state'], selected_mandi, "Wheat")
            print(f"💰 Price at {selected_mandi}: {price_data['modal_price']} {price_data['unit']}")

    except Exception as e:
        print(f"❌ Error: {e}")

if __name__ == "__main__":
    import sys
    # Check if we are running in a Jupyter/IPython environment
    if 'ipykernel' in sys.modules:
        import asyncio
        loop = asyncio.get_event_loop()
        if loop.is_running():
            # In a notebook, we can't use asyncio.run. 
            # You should manually call 'await main()' in a cell.
            print("📝 Detected Jupyter environment. Please run 'await main()' in a new cell.")
        else:
            loop.run_until_complete(main())
    else:
        # Standalone script
        asyncio.run(main())
# --- EXAMPLE USAGE ---
async def main():
    service = MandiPriceService()
    f_id = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb" 
    
    try:
        # 1. Get Location
        ctx = await service.get_farmer_context(f_id)
        print(f"📍 Location: {ctx['district']}, {ctx['state']}")
        
        # 2. Get Mandis
        mandis = service.get_available_mandis(ctx['state'], ctx['district'])
        print(f"🏪 Available Mandis: {mandis}")
        
        # 3. Fetch Prices
        if mandis:
            selected_mandi = mandis[0]
            price_data = await service.fetch_live_prices(ctx['state'], selected_mandi, "Wheat")
            print(f"💰 Price at {selected_mandi}: {price_data['modal_price']} {price_data['unit']}")

    except Exception as e:
        print(f"❌ Error: {e}")

if __name__ == "__main__":
    import sys
    # Check if we are running in a Jupyter/IPython environment
    if 'ipykernel' in sys.modules:
        import asyncio
        loop = asyncio.get_event_loop()
        if loop.is_running():
            # In a notebook, we can't use asyncio.run. 
            # You should manually call 'await main()' in a cell.
            print("📝 Detected Jupyter environment. Please run 'await main()' in a new cell.")
        else:
            loop.run_until_complete(main())
    else:
        # Standalone script
        asyncio.run(main())
# --- EXAMPLE USAGE ---
async def main():
    service = MandiPriceService()
    f_id = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb" 
    
    try:
        # 1. Get Location
        ctx = await service.get_farmer_context(f_id)
        print(f"📍 Location: {ctx['district']}, {ctx['state']}")
        
        # 2. Get Mandis
        mandis = service.get_available_mandis(ctx['state'], ctx['district'])
        print(f"🏪 Available Mandis: {mandis}")
        
        # 3. Fetch Prices
        if mandis:
            selected_mandi = mandis[0]
            price_data = await service.fetch_live_prices(ctx['state'], selected_mandi, "Wheat")
            print(f"💰 Price at {selected_mandi}: {price_data['modal_price']} {price_data['unit']}")

    except Exception as e:
        print(f"❌ Error: {e}")

if __name__ == "__main__":
    import sys
    # Check if we are running in a Jupyter/IPython environment
    if 'ipykernel' in sys.modules:
        import asyncio
        loop = asyncio.get_event_loop()
        if loop.is_running():
            # In a notebook, we can't use asyncio.run. 
            # You should manually call 'await main()' in a cell.
            print("📝 Detected Jupyter environment. Please run 'await main()' in a new cell.")
        else:
            loop.run_until_complete(main())
    else:
        # Standalone script
        asyncio.run(main())


📝 Detected Jupyter environment. Please run 'await main()' in a new cell.
📝 Detected Jupyter environment. Please run 'await main()' in a new cell.
📝 Detected Jupyter environment. Please run 'await main()' in a new cell.


In [15]:
# --- EXAMPLE USAGE ---
async def main():
    service = MandiPriceService()
    f_id = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb" 
    
    try:
        # 1. Get Location
        ctx = await service.get_farmer_context(f_id)
        print(f"📍 Location: {ctx['district']}, {ctx['state']}")
        
        # 2. Get Mandis
        mandis = service.get_available_mandis(ctx['state'], ctx['district'])
        print(f"🏪 Available Mandis: {mandis}")
        
        # 3. Fetch Prices
        if mandis:
            selected_mandi = mandis[0]
            price_data = await service.fetch_live_prices(ctx['state'], selected_mandi, "Wheat")
            print(f"💰 Price at {selected_mandi}: {price_data['modal_price']} {price_data['unit']}")

    except Exception as e:
        print(f"❌ Error: {e}")

if __name__ == "__main__":
    import sys
    # Check if we are running in a Jupyter/IPython environment
    if 'ipykernel' in sys.modules:
        import asyncio
        loop = asyncio.get_event_loop()
        if loop.is_running():
            # In a notebook, we can't use asyncio.run. 
            # You should manually call 'await main()' in a cell.
            print("📝 Detected Jupyter environment. Please run 'await main()' in a new cell.")
        else:
            loop.run_until_complete(main())
    else:
        # Standalone script
        asyncio.run(main())


📝 Detected Jupyter environment. Please run 'await main()' in a new cell.


In [24]:
import os
import json
import httpx
import asyncpg
import asyncio
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")
MANDI_JSON_PATH ="D:\CA_content\Python\KissanSathi\krishisarthi-api\mandi_master.json"
AGMARKNET_API_KEY = os.getenv("AGMARKNET_API_KEY")
AGMARKNET_URL = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"

class MandiPriceService:
    def __init__(self):
        self.mandi_data = self._load_mandi_data()

    def _load_mandi_data(self):
        """Loads the mandi_master.json file safely."""
        # Using absolute path check for reliability
        actual_path = MANDI_JSON_PATH.strip('"').strip("'")
        if not os.path.exists(actual_path):
            # Fallback to local directory if env path fails
            actual_path = "mandi_master.json"
            
        if not os.path.exists(actual_path):
            print(f"⚠️ Warning: Mandi JSON file not found at {actual_path}")
            return {}
            
        with open(actual_path, "r", encoding="utf-8") as f:
            return json.load(f)

    async def get_farmer_context(self, farmer_id: str):
        """Step 1: Fetch farmer's state and district from the database."""
        try:
            conn = await asyncpg.connect(DATABASE_URL)
            row = await conn.fetchrow(
                "SELECT state_name, dist_name FROM farmers WHERE id = $1",
                farmer_id
            )
            await conn.close()
            
            if not row:
                raise Exception(f"Farmer with ID {farmer_id} not found in database.")
                
            return {
                "state": row["state_name"],
                "district": row["dist_name"]
            }
        except Exception as e:
            raise Exception(f"Database Error: {str(e)}")

    def get_available_mandis(self, state: str, district: str):
        """Step 2: Get list of mandis from JSON using case-insensitive matching."""
        if not self.mandi_data:
            return []

        # 1. Try exact match first
        state_data = self.mandi_data.get(state)
        if state_data:
            mandis = state_data.get(district)
            if mandis:
                return mandis

        # 2. Case-insensitive fallback
        target_state_data = None
        for s_key in self.mandi_data:
            if s_key.lower() == state.lower():
                target_state_data = self.mandi_data[s_key]
                break
        
        if target_state_data:
            for d_key in target_state_data:
                if d_key.lower() == district.lower():
                    return target_state_data[d_key]
        
        return []

    async def fetch_live_prices(self, state: str, mandi_name: str, crop: str):
        """Step 3: Call Agmarknet API and return the latest price result."""
        if not AGMARKNET_API_KEY:
            raise Exception("API Configuration Error: AGMARKNET_API_KEY is not set.")

        params = {
            "api-key": AGMARKNET_API_KEY,
            "format": "json",
            "filters[state.keyword]": state,
            "filters[market]": mandi_name,
            "filters[commodity]": crop
        }

        async with httpx.AsyncClient() as client:
            response = await client.get(AGMARKNET_URL, params=params, timeout=15.0)
        
        if response.status_code != 200:
            raise Exception(f"Agmarknet API Error: Status {response.status_code} - {response.text}")

        data = response.json()
        if not data.get("records") or len(data["records"]) == 0:
            raise Exception(f"No price records found for '{crop}' at '{mandi_name}' in {state}.")

        return data["records"][0]

# --- ENTRY POINT ---
async def main():
    """Main execution flow."""
    service = MandiPriceService()
    
    # Replace with an actual farmer_id from your database
    test_farmer_id = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb" 
    test_crop = "Wheat"
    
    print(f"🚀 Starting Mandi Price Fetch for Farmer: {test_farmer_id}")
    
    try:
        # Step 1: Get location from DB
        ctx = await service.get_farmer_context(test_farmer_id)
        print(f"📍 Database Location: {ctx['district']}, {ctx['state']}")
        
        # Step 2: Get Mandis from JSON
        mandis = service.get_available_mandis(ctx['state'], ctx['district'])
        print(f"🏪 Available APMCs (from JSON): {mandis}")
        
        # Step 3: Fetch Price for the first available mandi
        if mandis:
            selected_mandi = mandis[0]
            print(f"📡 Fetching live prices for {test_crop} at {selected_mandi}...")
            price_data = await service.fetch_live_prices(ctx['state'], selected_mandi, test_crop)
            
            print("\n✅ PRICE DETAILS:")
            print("-" * 30)
            print(f"  Mandi       : {price_data.get('market')}")
            print(f"  Commodity   : {price_data.get('commodity')}")
            print(f"  Modal Price : {price_data.get('modal_price')} {price_data.get('unit', 'per quintal')}")
            print(f"  Date        : {price_data.get('arrival_date')}")
            print("-" * 30)
        else:
            print(f"⚠️ No matching mandis found in master data for {ctx['district']}.")

    except Exception as e:
        print(f"❌ Error occurred: {e}")

if __name__ == "__main__":
    # Handle running in Jupyter vs Script
    try:
        import sys
        if 'ipykernel' in sys.modules:
            # We are in a notebook; use await directly
            print("Running in Notebook environment...")
            # Note: In a notebook cell, you should just write: await main()
            # This block is here for compatibility.
            loop = asyncio.get_event_loop()
            if loop.is_running():
                print("⚠️ Event loop is already running. Please run 'await main()' in a new cell.")
            else:
                loop.run_until_complete(main())
        else:
            # Standalone script
            asyncio.run(main())
    except Exception as e:
        # Fallback if asyncio.run fails due to existing loop
        if "asyncio.run() cannot be called from a running event loop" in str(e):
             print("⚠️ Please run 'await main()' instead of running the whole script.")
        else:
            raise e


Running in Notebook environment...
⚠️ Event loop is already running. Please run 'await main()' in a new cell.


<>:12: SyntaxWarning: invalid escape sequence '\C'
<>:12: SyntaxWarning: invalid escape sequence '\C'
C:\Users\asus\AppData\Local\Temp\ipykernel_9412\3132993798.py:12: SyntaxWarning: invalid escape sequence '\C'
  MANDI_JSON_PATH ="D:\CA_content\Python\KissanSathi\krishisarthi-api\mandi_master.json"


In [25]:
await main()


🚀 Starting Mandi Price Fetch for Farmer: c59f6f44-1a98-4eaa-8cf0-3581316a32bb
📍 Database Location: agra, Uttar Pradesh
🏪 Available APMCs (from JSON): ['Achnera APMC', 'Agra APMC', 'Fatehabad APMC', 'Fatehpur Sikri APMC', 'Jagnair APMC', 'Jarar APMC']
📡 Fetching live prices for Wheat at Achnera APMC...
❌ Error occurred: 


## Report

In [30]:
import os
import json
import httpx
import asyncpg
import asyncio
import traceback  # Added for detailed error reporting
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")
MANDI_JSON_PATH ="D:\CA_content\Python\KissanSathi\krishisarthi-api\mandi_master.json"
AGMARKNET_API_KEY = os.getenv("AGMARKNET_API_KEY")
AGMARKNET_URL = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"

class MandiPriceService:
    def __init__(self):
        self.mandi_data = self._load_mandi_data()

    def _load_mandi_data(self):
        actual_path = MANDI_JSON_PATH.strip('"').strip("'")
        if not os.path.exists(actual_path):
            actual_path = "mandi_master.json"
        if not os.path.exists(actual_path):
            return {}
        with open(actual_path, "r", encoding="utf-8") as f:
            return json.load(f)

    async def get_farmer_context(self, farmer_id: str):
        try:
            conn = await asyncpg.connect(DATABASE_URL)
            row = await conn.fetchrow("SELECT state_name, dist_name FROM farmers WHERE id = $1", farmer_id)
            await conn.close()
            if not row: raise Exception(f"Farmer ID {farmer_id} not found.")
            return {"state": row["state_name"], "district": row["dist_name"]}
        except Exception as e:
            raise Exception(f"DB Error: {str(e)}")

    def get_available_mandis(self, state: str, district: str):
        target_state_data = None
        for s_key in self.mandi_data:
            if s_key.lower() == state.lower():
                target_state_data = self.mandi_data[s_key]
                break
        if target_state_data:
            for d_key in target_state_data:
                if d_key.lower() == district.lower():
                    return target_state_data[d_key]
        return []

    async def fetch_live_prices(self, state: str, mandi_name: str, crop: str):
        """Step 3: Call Agmarknet API with better error reporting and name cleaning."""
        if not AGMARKNET_API_KEY:
            raise Exception("API Configuration Error: AGMARKNET_API_KEY is not set.")

        # CLEANING: Agmarknet API usually doesn't have " APMC" in its names
        clean_mandi = mandi_name.replace(" APMC", "").strip()
        
        params = {
            "api-key": AGMARKNET_API_KEY,
            "format": "json",
            "filters[state]": state,      # Changed from state.keyword for better compatibility
            "filters[market]": clean_mandi,
            "filters[commodity]": crop
        }

        print(f"🔍 Debug: Calling API with params: {params}")

        try:
            async with httpx.AsyncClient() as client:
                response = await client.get(AGMARKNET_URL, params=params, timeout=15.0)
            
            if response.status_code != 200:
                # Log the actual response body to see the error from the server
                print(f"❌ API Server Error: {response.text}")
                raise Exception(f"Agmarknet API returned status {response.status_code}")

            data = response.json()
            if not data.get("records") or len(data["records"]) == 0:
                # Try one more time without the state filter (sometimes states are indexed differently)
                params.pop("filters[state]", None)
                print("🔄 No records found with state filter. Retrying with market/commodity only...")
                async with httpx.AsyncClient() as client:
                    response = await client.get(AGMARKNET_URL, params=params, timeout=15.0)
                data = response.json()

            if not data.get("records") or len(data["records"]) == 0:
                raise Exception(f"No price records found for '{crop}' at '{clean_mandi}' in {state}.")

            return data["records"][0]

        except Exception as e:
            # Re-raise with more context
            raise Exception(f"Fetch failed: {str(e)}")
# --- ENTRY POINT ---
async def main():
    service = MandiPriceService()
    test_farmer_id = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb" 
    test_crop = "Wheat"
    
    try:
        ctx = await service.get_farmer_context(test_farmer_id)
        mandis = service.get_available_mandis(ctx['state'], ctx['district'])
        
        if mandis:
            selected_mandi = mandis[1]
            print(f"📡 Fetching live prices for {test_crop} at {selected_mandi}...")
            price_data = await service.fetch_live_prices(ctx['state'], selected_mandi, test_crop)
            
            # Using .get() with defaults to avoid KeyError
            market = price_data.get('market', selected_mandi)
            price = price_data.get('modal_price', 'N/A')
            date = price_data.get('arrival_date', 'N/A')
            
            print(f"\n✅ SUCCESS: {market}")
            print("-" * 30)
            print(f"  Commodity   : {test_crop}")
            print(f"  Modal Price : ₹{price} per Quintal") # Explicitly defined unit
            print(f"  Arrival Date: {date}")
            print("-" * 30)
        else:
            print(f"⚠️ No mandis found for {ctx['district']}.")

    except Exception:
        print("\n💥 DETAILED ERROR REPORT:")
        print("-" * 50)
        traceback.print_exc()
        print("-" * 50)


<>:13: SyntaxWarning: invalid escape sequence '\C'
<>:13: SyntaxWarning: invalid escape sequence '\C'
C:\Users\asus\AppData\Local\Temp\ipykernel_9412\2694589470.py:13: SyntaxWarning: invalid escape sequence '\C'
  MANDI_JSON_PATH ="D:\CA_content\Python\KissanSathi\krishisarthi-api\mandi_master.json"


In [31]:
await main()

📡 Fetching live prices for Wheat at Agra APMC...
🔍 Debug: Calling API with params: {'api-key': '[REDACTED_AGMARKNET_KEY]', 'format': 'json', 'filters[state]': 'Uttar Pradesh', 'filters[market]': 'Agra', 'filters[commodity]': 'Wheat'}
🔄 No records found with state filter. Retrying with market/commodity only...

💥 DETAILED ERROR REPORT:
--------------------------------------------------
--------------------------------------------------


Traceback (most recent call last):
  File "C:\Users\asus\AppData\Local\Temp\ipykernel_9412\2694589470.py", line 89, in fetch_live_prices
    raise Exception(f"No price records found for '{crop}' at '{clean_mandi}' in {state}.")
Exception: No price records found for 'Wheat' at 'Agra' in Uttar Pradesh.

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\asus\AppData\Local\Temp\ipykernel_9412\2694589470.py", line 109, in main
    price_data = await service.fetch_live_prices(ctx['state'], selected_mandi, test_crop)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\asus\AppData\Local\Temp\ipykernel_9412\2694589470.py", line 95, in fetch_live_prices
    raise Exception(f"Fetch failed: {str(e)}")
Exception: Fetch failed: No price records found for 'Wheat' at 'Agra' in Uttar Pradesh.


In [1]:
import os
import json
import httpx
import asyncpg
import asyncio
import traceback
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")
MANDI_JSON_PATH = r"D:\CA_content\Python\KissanSathi\krishisarthi-api\mandi_master.json"
AGMARKNET_API_KEY = os.getenv("AGMARKNET_API_KEY")
AGMARKNET_URL = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"

class MandiPriceService:
    def __init__(self):
        self.mandi_data = self._load_mandi_data()

    def _load_mandi_data(self):
        actual_path = MANDI_JSON_PATH.strip('"').strip("'")
        if not os.path.exists(actual_path):
            actual_path = "mandi_master.json"
        if not os.path.exists(actual_path):
            return {}
        with open(actual_path, "r", encoding="utf-8") as f:
            return json.load(f)

    async def get_farmer_context(self, farmer_id: str):
        try:
            conn = await asyncpg.connect(DATABASE_URL)
            row = await conn.fetchrow("SELECT state_name, dist_name FROM farmers WHERE id = $1", farmer_id)
            await conn.close()
            if not row: raise Exception(f"Farmer ID {farmer_id} not found.")
            return {"state": row["state_name"], "district": row["dist_name"]}
        except Exception as e:
            raise Exception(f"DB Error: {str(e)}")

    def get_available_mandis(self, state: str, district: str):
        target_state_data = None
        for s_key in self.mandi_data:
            if s_key.lower() == state.lower():
                target_state_data = self.mandi_data[s_key]
                break
        if target_state_data:
            for d_key in target_state_data:
                if d_key.lower() == district.lower():
                    return target_state_data[d_key]
        return []

    async def fetch_live_prices(self, state: str, mandi_name: str, crop: str):
        if not AGMARKNET_API_KEY:
            raise Exception("API Configuration Error: AGMARKNET_API_KEY is not set.")

        clean_mandi = mandi_name.replace(" APMC", "").strip()
        params = {
            "api-key": AGMARKNET_API_KEY,
            "format": "json",
            "filters[state]": state,
            "filters[market]": clean_mandi,
            "filters[commodity]": crop
        }

        async with httpx.AsyncClient() as client:
            response = await client.get(AGMARKNET_URL, params=params, timeout=15.0)
        
        if response.status_code != 200:
            raise Exception(f"Agmarknet API returned status {response.status_code}")

        data = response.json()
        if not data.get("records") or len(data["records"]) == 0:
            raise Exception(f"No price records found for '{crop}' at '{clean_mandi}' in {state}.")

        return data["records"][0]

# --- INTERACTIVE JUPYTER UI ---
def create_mandi_dropdown(farmer_id, crop="Wheat"):
    """Creates a dropdown and button UI for Jupyter Notebooks."""
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    
    service = MandiPriceService()
    output = widgets.Output()
    
    async def load_and_display():
        with output:
            print("⏳ Loading mandis for farmer...")
            try:
                ctx = await service.get_farmer_context(farmer_id)
                mandis = service.get_available_mandis(ctx['state'], ctx['district'])
                
                if not mandis:
                    print(f"⚠️ No mandis found for {ctx['district']}.")
                    return

                # Create Dropdown and Button
                mandi_dropdown = widgets.Dropdown(
                    options=mandis,
                    description='Select Mandi:',
                    style={'description_width': 'initial'},
                    layout={'width': 'max-content'}
                )
                
                fetch_button = widgets.Button(
                    description="Get Live Price",
                    button_style='success',
                    icon='search'
                )

                def on_click(b):
                    with output:
                        clear_output()
                        # Re-display the dropdown and button
                        display(mandi_dropdown, fetch_button)
                        selected = mandi_dropdown.value
                        print(f"\n📡 Fetching price for {crop} at {selected}...")
                        
                        # Run async fetch
                        async def do_fetch():
                            try:
                                price_data = await service.fetch_live_prices(ctx['state'], selected, crop)
                                print(f"\n✅ SUCCESS: {price_data.get('market')}")
                                print(f"💰 Modal Price: ₹{price_data.get('modal_price')} per Quintal")
                                print(f"📅 Date: {price_data.get('arrival_date')}")
                            except Exception as e:
                                print(f"❌ Error: {e}")
                        
                        asyncio.create_task(do_fetch())

                fetch_button.on_click(on_click)
                
                clear_output()
                display(mandi_dropdown, fetch_button)

            except Exception as e:
                print(f"❌ Initialization Error: {e}")

    # Start the loading process
    asyncio.create_task(load_and_display())
    return output

# --- ENTRY POINT ---
async def main():
    """Default non-interactive execution."""
    # (Same logic as before for standard scripts)
    service = MandiPriceService()
    test_farmer_id = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb"
    # ... code here ...

if __name__ == "__main__":
    import sys
    if 'ipykernel' in sys.modules:
        # IN JUPYTER: Just call the dropdown function
        display_output = create_mandi_dropdown("c59f6f44-1a98-4eaa-8cf0-3581316a32bb", "Wheat")
        from IPython.display import display
        display(display_output)
    else:
        asyncio.run(main())


Output()

In [3]:
import os
import json
import httpx
import asyncpg
import asyncio
import traceback
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")
MANDI_JSON_PATH = r"D:\CA_content\Python\KissanSathi\krishisarthi-api\mandi_master.json"
AGMARKNET_API_KEY = os.getenv("AGMARKNET_API_KEY")
AGMARKNET_URL = "https://api.data.gov.in/resource/9ef84268-d588-465a-a308-a864a43d0070"

class MandiPriceService:
    def __init__(self):
        self.mandi_data = self._load_mandi_data()

    def _load_mandi_data(self):
        actual_path = MANDI_JSON_PATH.strip('"').strip("'")
        if not os.path.exists(actual_path):
            actual_path = "mandi_master.json"
        if not os.path.exists(actual_path):
            return {}
        with open(actual_path, "r", encoding="utf-8") as f:
            return json.load(f)

    async def get_farmer_context(self, farmer_id: str):
        try:
            conn = await asyncpg.connect(DATABASE_URL)
            row = await conn.fetchrow("SELECT state_name, dist_name FROM farmers WHERE id = $1", farmer_id)
            await conn.close()
            if not row: raise Exception(f"Farmer ID {farmer_id} not found.")
            return {"state": row["state_name"], "district": row["dist_name"]}
        except Exception as e:
            raise Exception(f"DB Error: {str(e)}")

    def get_available_mandis(self, state: str, district: str):
        target_state_data = None
        for s_key in self.mandi_data:
            if s_key.lower() == state.lower():
                target_state_data = self.mandi_data[s_key]
                break
        if target_state_data:
            for d_key in target_state_data:
                if d_key.lower() == district.lower():
                    return target_state_data[d_key]
        return []

    async def fetch_live_prices(self, state: str, mandi_name: str, crop: str):
        if not AGMARKNET_API_KEY:
            raise Exception("API Configuration Error: AGMARKNET_API_KEY is not set.")

        clean_mandi = mandi_name.replace(" APMC", "").strip()
        params = {
            "api-key": AGMARKNET_API_KEY,
            "format": "json",
            "filters[state]": state,
            "filters[market]": clean_mandi,
            "filters[commodity]": crop
        }

        async with httpx.AsyncClient() as client:
            response = await client.get(AGMARKNET_URL, params=params, timeout=15.0)
        
        if response.status_code != 200:
            raise Exception(f"Agmarknet API returned status {response.status_code}")

        data = response.json()
        if not data.get("records") or len(data["records"]) == 0:
            raise Exception(f"No price records found for '{crop}' at '{clean_mandi}' in {state}.")

        return data["records"][0]

# --- MAIN SEQUENTIAL FLOW ---
async def main():
    service = MandiPriceService()
    test_farmer_id = "c59f6f44-1a98-4eaa-8cf0-3581316a32bb" 
    test_crop = "Wheat"
    
    print(f"🚀 Initializing search for Farmer: {test_farmer_id}")
    
    try:
        # Step 1: Get location
        ctx = await service.get_farmer_context(test_farmer_id)
        print(f"📍 Location detected: {ctx['district']}, {ctx['state']}")
        
        # Step 2: Get mandis from JSON
        mandis = service.get_available_mandis(ctx['state'], ctx['district'])
        
        if not mandis:
            print(f"⚠️ No mandis found for {ctx['district']}.")
            return

        # Step 3: Display options to user
        print(f"\n🏪 Available APMCs in {ctx['district']}:")
        for i, m in enumerate(mandis):
            print(f"  {i}. {m}")
            
        # Step 4: Get selection
        choice = input(f"\nEnter the number (0-{len(mandis)-1}) for the mandi you want: ")
        
        try:
            idx = int(choice)
            if 0 <= idx < len(mandis):
                selected_mandi = mandis[idx]
                print(f"\n📡 Fetching live prices for {test_crop} at {selected_mandi}...")
                
                # Step 5: Fetch price
                price_data = await service.fetch_live_prices(ctx['state'], selected_mandi, test_crop)
                
                print(f"\n✅ SUCCESS: {price_data.get('market')}")
                print("-" * 30)
                print(f"  Commodity   : {test_crop}")
                print(f"  Modal Price : ₹{price_data.get('modal_price')} per Quintal")
                print(f"  Arrival Date: {price_data.get('arrival_date')}")
                print("-" * 30)
            else:
                print("❌ Error: Invalid number selected.")
        except ValueError:
            print("❌ Error: Please enter a valid number.")

    except Exception:
        print("\n💥 ERROR DETAILS:")
        traceback.print_exc()

if __name__ == "__main__":
    import sys
    if 'ipykernel' in sys.modules:
        print("📝 Please run 'await main()' in a new cell.")
    else:
        asyncio.run(main())


📝 Please run 'await main()' in a new cell.


In [7]:
await main()

🚀 Initializing search for Farmer: c59f6f44-1a98-4eaa-8cf0-3581316a32bb
📍 Location detected: agra, Uttar Pradesh

🏪 Available APMCs in agra:
  0. Achnera APMC
  1. Agra APMC
  2. Fatehabad APMC
  3. Fatehpur Sikri APMC
  4. Jagnair APMC
  5. Jarar APMC

📡 Fetching live prices for Wheat at Achnera APMC...

✅ SUCCESS: Achnera APMC
------------------------------
  Commodity   : Wheat
  Modal Price : ₹2350 per Quintal
  Arrival Date: 21/04/2026
------------------------------
